## Classe `BaslerCamera`
O que representa esta classe? (responda abaixo)


### Método `BaslerCamera.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, ip_address: Optional[str]=None): #inicia um dicionario para os metodos da pasta que vao ser chamados e importados como biblioteca 
        self.ip_address = ip_address # localmente cria a variavel ip_address
        self._camera = None # localmente cria a variavel vazia camera
        self._initialized = False # localmente cria a variavel vazia initialized
        log.info(f" Câmera Basler criada (IP: {ip_address or 'auto'})") # utiliza a funcao log para emitir a info com uma string de mensagem e a variavel ip adress caso seja manualmente setado ou auto 

### Método `BaslerCamera.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        if not BASLER_AVAILABLE:
            log.error('Pypylon NAO esta instalado! Instale com: pip install pypylon')
            raise RuntimeError('Biblioteca pypylon não disponível. Instale com: pip install pypylon')
        try:
            log.info(' Inicializando câmera Basler...')
            log.debug(f"  IP especificado: {self.ip_address or 'automático (primeira disponível)'}")
            tl_factory = pylon.TlFactory.GetInstance()
            log.debug('  Fábrica de dispositivos obtida')
            devices = tl_factory.EnumerateDevices()
            log.info(f'   Encontradas {len(devices)} câmera(s) Basler')
            if not devices:
                msg = 'Nenhuma camera Basler encontrada.'
                log.error(f'ERRO: {msg}')
                raise RuntimeError(msg)
            for i, device in enumerate(devices):
                model = device.GetModelName() if hasattr(device, 'GetModelName') else 'Desconhecido'
                ip = device.GetIpAddress() if hasattr(device, 'GetIpAddress') else 'N/A'
                log.info(f'     [{i}] {model} (IP: {ip})')
            selected_device = None
            if self.ip_address:
                log.debug(f'  Procurando câmera com IP: {self.ip_address}')
                for device in devices:
                    if hasattr(device, 'GetIpAddress') and device.GetIpAddress() == self.ip_address:
                        selected_device = device
                        log.info(f'  OK - Camera encontrada com IP: {self.ip_address}')
                        break
                if not selected_device:
                    log.warning(f'    IP {self.ip_address} não encontrado, usando primeira câmera')
            if selected_device is None:
                selected_device = devices[0]
                model = selected_device.GetModelName() if hasattr(selected_device, 'GetModelName') else 'Desconhecido'
                log.info(f'   Usando primeira câmera: {model}')
            log.debug('  Criando instância da câmera...')
            self._camera = pylon.InstantCamera(tl_factory.CreateDevice(selected_device))
            log.debug('  Abrindo câmera...')
            self._camera.Open()
            log.debug('  OK - Camera aberta com sucesso')
            log.debug('  Configurando resolução máxima...')
            self._camera.Width.SetValue(self._camera.Width.Max)
            self._camera.Height.SetValue(self._camera.Height.Max)
            width = self._camera.Width.Value
            height = self._camera.Height.Value
            log.info(f'   Resolução: {width}x{height}')
            try:
                self._camera.ExposureAuto.SetValue('Continuous')
                log.info('    Exposição automática ativada')
            except Exception as e:
                log.warning(f'    Não foi possível configurar exposição automática: {e}')
            self._initialized = True
            log.info('OK - Camera Basler inicializada com sucesso!')
            return True
        except Exception as e:
            log.error(f'ERRO - Falha ao inicializar camera Basler: {type(e).__name__}: {e}')
            import traceback
            log.debug(traceback.format_exc())
            self._initialized = False
            return False

### Método `BaslerCamera.capture`
O que faz este método? (responda abaixo)


In [ ]:
def capture(self) -> Optional[np.ndarray]:
        if not self._initialized or self._camera is None:
            raise RuntimeError('Câmera não inicializada. Chame initialize() primeiro.')
        try:
            log.debug('Capturando imagem da Basler...')
            self._camera.StartGrabbingMax(1)
            grab_result = self._camera.RetrieveResult(5000, pylon.TimeoutHandling_ThrowException)
            if grab_result.GrabSucceeded():
                image = grab_result.Array
                if len(image.shape) == 2:
                    image = cv2.cvtColor(image, cv2.COLOR_BAYER_BG2BGR)
                grab_result.Release()
                log.debug(f'OK - Imagem capturada: {image.shape}')
                return image
            else:
                grab_result.Release()
                raise RuntimeError('Falha na captura da imagem')
        except Exception as e:
            log.error(f'ERRO - Erro ao capturar imagem: {e}')
            return None

### Método `BaslerCamera.release`
O que faz este método? (responda abaixo)


In [ ]:
def release(self):
        if self._camera is not None and self._camera.IsOpen():
            self._camera.Close()
            log.info(' Câmera Basler liberada')
        self._initialized = False

### Método `BaslerCamera.get_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_info(self) -> Dict[str, Any]:
        info = {'type': 'basler', 'ip': self.ip_address, 'initialized': self._initialized, 'available': BASLER_AVAILABLE}
        if self._camera and self._camera.IsOpen():
            info['model'] = self._camera.GetDeviceInfo().GetModelName()
            info['width'] = self._camera.Width.GetValue()
            info['height'] = self._camera.Height.GetValue()
        return info

### Método `BaslerCamera.is_available`
O que faz este método? (responda abaixo)


In [ ]:
def is_available(self) -> bool:
        return BASLER_AVAILABLE

### Método `BaslerCamera.get_parameters`
O que faz este método? (responda abaixo)


In [ ]:
def get_parameters(self) -> Dict[str, Any]:
        if not self._initialized or not self._camera or (not self._camera.IsOpen()):
            return {}
        try:
            return {'gain': {'value': float(self._camera.Gain.GetValue()) if hasattr(self._camera, 'Gain') else 0, 'type': 'float', 'label': 'Ganho', 'min': 0, 'max': 30, 'step': 0.5, 'description': 'Ganho da câmera (0 a 30 dB)'}, 'exposure_time': {'value': float(self._camera.ExposureTime.GetValue()) if hasattr(self._camera, 'ExposureTime') else 0, 'type': 'float', 'label': 'Tempo de Exposição (µs)', 'min': 26, 'max': 30000000, 'step': 1000, 'description': 'Tempo de exposição em microsegundos (26 µs a 30s)'}, 'frame_rate': {'value': float(self._camera.AcquisitionFrameRate.GetValue()) if hasattr(self._camera, 'AcquisitionFrameRate') else 30, 'type': 'float', 'label': 'Taxa de Quadros (FPS)', 'min': 1, 'max': 200, 'step': 1, 'description': 'Taxa de aquisição em quadros por segundo'}, 'width': {'value': int(self._camera.Width.GetValue()), 'type': 'int', 'label': 'Largura', 'min': 100, 'max': 2048, 'step': 4, 'description': 'Largura da imagem em pixels'}, 'height': {'value': int(self._camera.Height.GetValue()), 'type': 'int', 'label': 'Altura', 'min': 100, 'max': 2048, 'step': 4, 'description': 'Altura da imagem em pixels'}, 'balance_white': {'value': 'auto' if hasattr(self._camera, 'BalanceWhiteAuto') and self._camera.BalanceWhiteAuto.GetValue() == 1 else 'manual', 'type': 'enum', 'label': 'Balanço de Branco', 'options': ['manual', 'auto'], 'description': 'Modo de balanço de branco'}}
        except Exception as e:
            log.debug(f'Erro ao ler parâmetros Basler: {e}')
            return {}

### Método `BaslerCamera.set_parameter`
O que faz este método? (responda abaixo)


In [ ]:
def set_parameter(self, param_name: str, value: Any) -> bool:
        if not self._initialized or not self._camera or (not self._camera.IsOpen()):
            return False

        def _try_set(node, val) -> bool:

            def _is_writable(n) -> bool:
                try:
                    if hasattr(n, 'GetAccessMode'):
                        mode = n.GetAccessMode()
                        if mode not in (pylon.RW, pylon.WO):
                            return False
                    if hasattr(n, 'IsWritable'):
                        return bool(n.IsWritable())
                except Exception:
                    pass
                return True
            if not _is_writable(node):
                log.debug(f'Basler node not writable, skipping')
                return False
            try:
                node.SetValue(val)
                return True
            except Exception as exc:
                msg = str(exc)
                if 'Node not existing' in msg or 'Expected a string' in msg:
                    log.debug(f'Basler node skipped ({msg})')
                    if 'Expected a string' in msg:
                        try:
                            node.SetValue(str(val))
                            return True
                        except Exception as e2:
                            log.debug(f'Retry with str also failed: {e2}')
                    return False
                raise
        try:
            if param_name == 'gain' and hasattr(self._camera, 'Gain'):
                return _try_set(self._camera.Gain, float(value))
            elif param_name == 'exposure_time' and hasattr(self._camera, 'ExposureTime'):
                return _try_set(self._camera.ExposureTime, int(value))
            elif param_name == 'frame_rate' and hasattr(self._camera, 'AcquisitionFrameRate'):
                return _try_set(self._camera.AcquisitionFrameRate, float(value))
            elif param_name == 'width' and hasattr(self._camera, 'Width'):
                return _try_set(self._camera.Width, int(value))
            elif param_name == 'height' and hasattr(self._camera, 'Height'):
                return _try_set(self._camera.Height, int(value))
            elif param_name == 'balance_white' and hasattr(self._camera, 'BalanceWhiteAuto'):
                mode_val = 1 if str(value).lower() == 'auto' else 0
                return _try_set(self._camera.BalanceWhiteAuto, mode_val)

            def to_pascal(s: str) -> str:
                parts = s.split('_')
                return ''.join((p.capitalize() for p in parts if p))
            tried_names = []
            for candidate in (param_name, to_pascal(param_name)):
                tried_names.append(candidate)
                if hasattr(self._camera, candidate):
                    node = getattr(self._camera, candidate)
                    if _try_set(node, value):
                        log.info(f' Basler: {candidate} = {value} (generic)')
                        return True
            log.debug(f'Parâmetro Basler não reconhecido: {param_name} (tentadas: {tried_names})')
            return False
        except Exception as e:
            msg = str(e)
            if 'Node not existing' in msg or 'Expected a string' in msg:
                log.debug(f'Parâmetro Basler indisponível neste modelo: {msg}')
                return False
            log.error(f'Erro ao ajustar Basler: {e}')
            return False

### Método `BaslerCamera.test_connection`
O que faz este método? (responda abaixo)


In [ ]:
def test_connection(self) -> bool:
        if not BASLER_AVAILABLE:
            log.debug('  Pypylon não disponível')
            return False
        try:
            tl_factory = pylon.TlFactory.GetInstance()
            devices = tl_factory.EnumerateDevices()
            available = len(devices) > 0
            if available:
                log.debug(f'  OK - Basler detectada ({len(devices)} camera(s))')
            else:
                log.debug('  ERRO - Nenhuma Basler detectada')
            return available
        except Exception as e:
            log.debug(f'  Erro ao testar Basler: {e}')
            return False

### Método `BaslerCamera.apply_pfs_file`
O que faz este método? (responda abaixo)


In [ ]:
def apply_pfs_file(self, file_path: str) -> bool:
        if not BASLER_AVAILABLE:
            log.error('Pypylon não disponível, não é possível aplicar .pfs')
            return False
        if not self._initialized or not self._camera or (not self._camera.IsOpen()):
            log.error('Câmera não inicializada, não é possível aplicar .pfs')
            return False
        try:
            pylon.FeaturePersistence.Load(str(file_path), self._camera.GetNodeMap(), True)
            log.info(f'Basler .pfs aplicado via FeaturePersistence: {file_path}')
            return True
        except Exception as e:
            log.error(f'Erro ao aplicar .pfs via pylon: {e}')
            return False

## Classe `BaslerProfileManager`
O que representa esta classe? (responda abaixo)


### Método `BaslerProfileManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, profiles_dir: Optional[Path]=None):
        super().__init__(profiles_dir)
        self.parser = BaslerProfileParser()
        log.info('📁 Gerenciador de perfis Basler (.pfs) inicializado')

### Método `BaslerProfileManager.load_pfs_profile`
O que faz este método? (responda abaixo)


In [ ]:
def load_pfs_profile(self, pfs_file_path: Path, profile_name: Optional[str]=None) -> bool:
        try:
            if not pfs_file_path.exists():
                log.error(f'Arquivo .pfs não encontrado: {pfs_file_path}')
                return False
            parsed_data = self.parser.parse_pfs_file(pfs_file_path)
            if not parsed_data:
                log.error(f'Falha ao parsear arquivo .pfs: {pfs_file_path}')
                return False
            if profile_name is None:
                profile_name = pfs_file_path.stem
            json_profile = self._pfs_to_json_profile(parsed_data, profile_name)
            json_profile['_pfs_file'] = str(pfs_file_path)
            return self.create_profile(profile_name=profile_name, camera_type='basler', parameters=json_profile)
        except Exception as e:
            log.error(f'Erro ao carregar perfil .pfs: {e}')
            return False

### Método `BaslerProfileManager.save_pfs_profile`
O que faz este método? (responda abaixo)


In [ ]:
def save_pfs_profile(self, profile_name: str, pfs_file_path: Path) -> bool:
        try:
            profile_data = self.load_profile(profile_name)
            if not profile_data:
                log.error(f'Perfil não encontrado: {profile_name}')
                return False
            pfs_data = self._json_to_pfs_data(profile_data)
            return self.parser.save_pfs_file(pfs_file_path, pfs_data['parameters'], pfs_data['metadata'])
        except Exception as e:
            log.error(f'Erro ao salvar perfil .pfs: {e}')
            return False

### Método `BaslerProfileManager.modify_pfs_parameter`
O que faz este método? (responda abaixo)


In [ ]:
def modify_pfs_parameter(self, profile_name: str, param_name: str, value: str, selector: Optional[str]=None) -> bool:
        try:
            profile_data = self.load_profile(profile_name)
            if not profile_data:
                return False
            pfs_data = self._json_to_pfs_data(profile_data)
            self.parser.set_parameter_value(pfs_data, param_name, value, selector)
            updated_profile = self._pfs_to_json_profile(pfs_data, profile_name)
            profile_data['parameters'] = updated_profile
            profile_data['timestamp'] = __import__('datetime').datetime.now().isoformat()
            profile_path = self.profiles_dir / f'{profile_name}.json'
            with open(profile_path, 'w', encoding='utf-8') as f:
                json.dump(profile_data, f, indent=2, ensure_ascii=False)
            log.info(f'Parâmetro modificado: {param_name} = {value}')
            return True
        except Exception as e:
            log.error(f'Erro ao modificar parâmetro: {e}')
            return False

### Método `BaslerProfileManager.get_pfs_parameter`
O que faz este método? (responda abaixo)


In [ ]:
def get_pfs_parameter(self, profile_name: str, param_name: str, selector: Optional[str]=None) -> Optional[str]:
        try:
            profile_data = self.load_profile(profile_name)
            if not profile_data:
                return None
            pfs_data = self._json_to_pfs_data(profile_data)
            return self.parser.get_parameter_value(pfs_data, param_name, selector)
        except Exception as e:
            log.error(f'Erro ao obter parâmetro: {e}')
            return None

### Método `BaslerProfileManager.list_pfs_profiles`
O que faz este método? (responda abaixo)


In [ ]:
def list_pfs_profiles(self) -> List[str]:
        return self.list_profiles(camera_type='basler')

### Método `BaslerProfileManager.import_pfs_directory`
O que faz este método? (responda abaixo)


In [ ]:
def import_pfs_directory(self, pfs_directory: Path) -> int:
        if not pfs_directory.exists():
            log.error(f'Diretório não encontrado: {pfs_directory}')
            return 0
        imported_count = 0
        for pfs_file in pfs_directory.glob('*.pfs'):
            profile_name = pfs_file.stem
            if self.load_pfs_profile(pfs_file, profile_name):
                imported_count += 1
        log.info(f'Importados {imported_count} perfis .pfs')
        return imported_count

### Método `BaslerProfileManager.export_pfs_directory`
O que faz este método? (responda abaixo)


In [ ]:
def export_pfs_directory(self, export_directory: Path, profile_names: Optional[List[str]]=None) -> int:
        export_directory.mkdir(parents=True, exist_ok=True)
        if profile_names is None:
            profile_names = self.list_pfs_profiles()
        exported_count = 0
        for profile_name in profile_names:
            pfs_file_path = export_directory / f'{profile_name}.pfs'
            if self.save_pfs_profile(profile_name, pfs_file_path):
                exported_count += 1
        log.info(f'Exportados {exported_count} perfis .pfs')
        return exported_count

### Método `BaslerProfileManager._pfs_to_json_profile`
O que faz este método? (responda abaixo)


In [ ]:
def _pfs_to_json_profile(self, pfs_data: Dict[str, Any], profile_name: str) -> Dict[str, Any]:
        json_profile = pfs_data.get('parameters', {}).copy()
        json_profile.update({'format': 'pfs', 'metadata': pfs_data.get('metadata', {}), 'file_info': pfs_data.get('file_info', {}), 'converted_at': __import__('datetime').datetime.now().isoformat()})
        return json_profile

### Método `BaslerProfileManager._json_to_pfs_data`
O que faz este método? (responda abaixo)


In [ ]:
def _json_to_pfs_data(self, json_profile: Dict[str, Any]) -> Dict[str, Any]:
        profile_params = json_profile.get('parameters', {})
        parameters = {}
        metadata = profile_params.get('metadata', {})
        file_info = profile_params.get('file_info', {})
        for key, value in profile_params.items():
            if key not in ['format', 'metadata', 'file_info', 'converted_at']:
                parameters[key] = value
        return {'metadata': metadata, 'parameters': parameters, 'file_info': file_info}

### Método `BaslerProfileManager.create_pfs_profile_from_parameters`
O que faz este método? (responda abaixo)


In [ ]:
def create_pfs_profile_from_parameters(self, profile_name: str, parameters: Dict[str, Any], metadata: Optional[Dict[str, Any]]=None) -> bool:
        try:
            json_profile = parameters.copy()
            json_profile.update({'name': profile_name, 'format': 'pfs', 'metadata': metadata or {}, 'file_info': {}, 'converted_at': __import__('datetime').datetime.now().isoformat()})
            return self.create_profile(profile_name=profile_name, camera_type='basler', parameters=json_profile)
        except Exception as e:
            log.error(f'Erro ao criar perfil .pfs: {e}')
            return False

### Método `BaslerProfileManager.duplicate_pfs_profile`
O que faz este método? (responda abaixo)


In [ ]:
def duplicate_pfs_profile(self, source_profile: str, new_profile: str) -> bool:
        try:
            source_data = self.load_profile(source_profile)
            if not source_data:
                return False
            return self.create_profile(profile_name=new_profile, camera_type='basler', parameters=source_data['parameters'])
        except Exception as e:
            log.error(f'Erro ao duplicar perfil: {e}')
            return False

### Método `BaslerProfileManager.validate_pfs_profile`
O que faz este método? (responda abaixo)


In [ ]:
def validate_pfs_profile(self, profile_name: str) -> Dict[str, Any]:
        result = {'valid': False, 'errors': [], 'warnings': [], 'parameters_count': 0}
        try:
            profile_data = self.load_profile(profile_name)
            if not profile_data:
                result['errors'].append('Perfil não encontrado')
                return result
            parameters = profile_data.get('parameters', {})
            required_params = ['Width', 'Height', 'PixelFormat']
            recommended_params = ['ExposureTimeRaw', 'GainRaw', 'GammaEnable']
            result['parameters_count'] = len(parameters)
            for param in required_params:
                if param not in parameters:
                    result['errors'].append(f'Parâmetro obrigatório ausente: {param}')
            for param in recommended_params:
                if param not in parameters:
                    result['warnings'].append(f'Parâmetro recomendado ausente: {param}')
            if 'Width' in parameters and 'Height' in parameters:
                try:
                    width = int(parameters['Width'])
                    height = int(parameters['Height'])
                    if width <= 0 or height <= 0:
                        result['errors'].append('Dimensões inválidas')
                except ValueError:
                    result['errors'].append('Dimensões não são números válidos')
            if 'PixelFormat' in parameters:
                valid_formats = ['Mono8', 'Mono12', 'BayerRG8', 'BayerRG12', 'RGB8']
                if parameters['PixelFormat'] not in valid_formats:
                    result['warnings'].append(f"Formato de pixel não padrão: {parameters['PixelFormat']}")
            result['valid'] = len(result['errors']) == 0
        except Exception as e:
            result['errors'].append(f'Erro na validação: {str(e)}')
        return result

## Classe `BaslerProfileParser`
O que representa esta classe? (responda abaixo)


### Método `BaslerProfileParser.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self):
        self._log_prefix = '[BaslerProfileParser]'

### Método `BaslerProfileParser.parse_pfs_file`
O que faz este método? (responda abaixo)


In [ ]:
def parse_pfs_file(self, file_path: Path) -> Optional[Dict[str, Any]]:
        try:
            if not file_path.exists():
                log.error(f'{self._log_prefix} Arquivo não encontrado: {file_path}')
                return None
            parameters = {}
            metadata = {}
            with open(file_path, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue
                    if line.startswith('#'):
                        self._parse_comment(line, metadata)
                        continue
                    param = self._parse_parameter_line(line)
                    if param:
                        key, value = param
                        parameters[key] = value
                    else:
                        log.warning(f'{self._log_prefix} Linha inválida {line_num}: {line}')
            result = {'metadata': metadata, 'parameters': parameters, 'file_info': {'path': str(file_path), 'size': file_path.stat().st_size, 'modified': file_path.stat().st_mtime}}
            log.info(f'{self._log_prefix} Arquivo parseado: {len(parameters)} parâmetros')
            return result
        except Exception as e:
            log.error(f'{self._log_prefix} Erro ao parsear arquivo: {e}')
            return None

### Método `BaslerProfileParser._parse_comment`
O que faz este método? (responda abaixo)


In [ ]:
def _parse_comment(self, line: str, metadata: Dict[str, Any]):
        line = line.lstrip('#').strip()
        if line.startswith('{') and '}' in line:
            metadata['device_info'] = line
        elif 'GenApi persistence file' in line:
            version_match = re.search('version (\\d+\\.\\d+\\.\\d+)', line)
            if version_match:
                metadata['genapi_version'] = version_match.group(1)
        elif 'Device =' in line:
            device_match = re.search('Device = (.+)', line)
            if device_match:
                metadata['device'] = device_match.group(1).strip()
        elif 'Product GUID =' in line:
            guid_match = re.search('Product GUID = ([A-F0-9-]+)', line)
            if guid_match:
                metadata['product_guid'] = guid_match.group(1)

### Método `BaslerProfileParser._parse_parameter_line`
O que faz este método? (responda abaixo)


In [ ]:
def _parse_parameter_line(self, line: str) -> Optional[tuple]:
        parts = line.split('\t')
        if len(parts) < 2:
            return None
        param_name = parts[0].strip()
        if len(parts) >= 3 and parts[1].startswith('{') and ('}' in parts[1]):
            selector = parts[1].strip('{}')
            value = parts[2].strip()
            param_key = f'{param_name}@{{{selector}}}'
            return (param_key, value)
        else:
            value = parts[1].strip()
            return (param_name, value)

### Método `BaslerProfileParser.create_pfs_content`
O que faz este método? (responda abaixo)


In [ ]:
def create_pfs_content(self, parameters: Dict[str, Any], metadata: Optional[Dict[str, Any]]=None) -> str:
        lines = []
        lines.append('# {05D8C294-F295-4dfb-9D01-096BD04049F4}')
        lines.append('# GenApi persistence file (version 3.5.0)')
        if metadata:
            if 'device' in metadata:
                lines.append(f"# Device = {metadata['device']}")
            if 'product_guid' in metadata:
                lines.append(f"# Product GUID = {metadata['product_guid']}")
            if 'product_version_guid' in metadata:
                lines.append(f"# Product version GUID = {metadata['product_version_guid']}")
        lines.append('')
        for param_key, value in sorted(parameters.items()):
            if '@{' in param_key and param_key.endswith('}'):
                param_name, selector_part = param_key.split('@', 1)
                selector = selector_part.strip('{}')
                lines.append(f'{param_name}\t{{{selector}}}\t{value}')
            else:
                lines.append(f'{param_key}\t{value}')
        return '\n'.join(lines)

### Método `BaslerProfileParser.save_pfs_file`
O que faz este método? (responda abaixo)


In [ ]:
def save_pfs_file(self, file_path: Path, parameters: Dict[str, Any], metadata: Optional[Dict[str, Any]]=None) -> bool:
        try:
            content = self.create_pfs_content(parameters, metadata)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(content)
            log.info(f'{self._log_prefix} Arquivo salvo: {file_path}')
            return True
        except Exception as e:
            log.error(f'{self._log_prefix} Erro ao salvar arquivo: {e}')
            return False

### Método `BaslerProfileParser.get_parameter_value`
O que faz este método? (responda abaixo)


In [ ]:
def get_parameter_value(self, parsed_data: Dict[str, Any], param_name: str, selector: Optional[str]=None) -> Optional[str]:
        parameters = parsed_data.get('parameters', {})
        if selector:
            param_key = f'{param_name}@{{{selector}}}'
        else:
            param_key = param_name
        return parameters.get(param_key)

### Método `BaslerProfileParser.set_parameter_value`
O que faz este método? (responda abaixo)


In [ ]:
def set_parameter_value(self, parsed_data: Dict[str, Any], param_name: str, value: str, selector: Optional[str]=None):
        parameters = parsed_data.setdefault('parameters', {})
        if selector:
            param_key = f'{param_name}@{{{selector}}}'
        else:
            param_key = param_name
        parameters[param_key] = value

### Método `BaslerProfileParser.get_all_parameters_by_name`
O que faz este método? (responda abaixo)


In [ ]:
def get_all_parameters_by_name(self, parsed_data: Dict[str, Any], param_name: str) -> Dict[str, str]:
        parameters = parsed_data.get('parameters', {})
        result = {}
        for key, value in parameters.items():
            if key == param_name or key.startswith(f'{param_name}@{{'):
                if '@{' in key and key.endswith('}'):
                    selector = key.split('@', 1)[1]
                    result[selector] = value
                else:
                    result['default'] = value
        return result

## Classe `CameraManager`
O que representa esta classe? (responda abaixo)


### Método `CameraManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, config: Optional[Dict[str, Any]]=None):
        self.config = config or CAMERA_CONFIG
        self.cameras: List[ICamera] = []
        self.active_camera: Optional[ICamera] = None
        self._current_index = -1
        log.info(' Inicializando CameraManager...')
        self._create_camera_chain()

### Método `CameraManager._create_camera_chain`
O que faz este método? (responda abaixo)


In [ ]:
def _create_camera_chain(self):
        if 'primary' in self.config and BaslerCamera is not None:
            primary_config = self.config['primary']
            if primary_config['type'] == 'basler':
                try:
                    camera = BaslerCamera(ip_address=primary_config.get('ip'))
                    self.cameras.append(camera)
                    log.info(f' Adicionada câmera principal: Basler')
                except Exception as e:
                    log.warning(f'  Erro ao criar câmera Basler: {e}')
        elif 'primary' in self.config:
            log.warning('  Câmera primária (Basler) configurada mas não disponível')
        else:
            log.warning('  Nenhuma câmera primária configurada em settings.py')
        for i, fb_config in enumerate(self.config.get('fallbacks', [])):
            camera_type = fb_config['type']
            if camera_type == 'webcam':
                camera = WebcamCamera(camera_index=fb_config.get('index', 0))
                self.cameras.append(camera)
                log.info(f' Adicionada fallback {i + 1}: Webcam')
            elif camera_type == 'mock':
                camera = MockCamera(image_path=fb_config.get('image_path'))
                self.cameras.append(camera)
                log.info(f' Adicionada fallback {i + 1}: MockCamera')
        log.info(f' Total de câmeras na cadeia: {len(self.cameras)}')

### Método `CameraManager.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        max_retries = self.config.get('primary', {}).get('max_retries', 3)
        for i, camera in enumerate(self.cameras):
            log.info(f' Tentando inicializar câmera {i + 1}/{len(self.cameras)}: {camera.__class__.__name__}')
            for attempt in range(max_retries):
                try:
                    log.debug(f'  Tentativa {attempt + 1}/{max_retries}...')
                    if camera.initialize():
                        self.active_camera = camera
                        self._current_index = i
                        info = camera.get_info()
                        log.info(f' Câmera ativa: {camera.__class__.__name__}')
                        log.info(f"   Tipo: {info.get('type', 'N/A')}")
                        if 'model' in info:
                            log.info(f"   Modelo: {info['model']}")
                        if 'width' in info and 'height' in info:
                            log.info(f"   Resolução: {info['width']}x{info['height']}")
                        return True
                except Exception as e:
                    log.warning(f'  Tentativa {attempt + 1} falhou: {e}')
                    if attempt < max_retries - 1:
                        log.debug('  Aguardando 1 segundo antes de tentar novamente...')
                        import time
                        time.sleep(1)
                    else:
                        log.error(f' Todas as tentativas falharam para {camera.__class__.__name__}')
            log.warning(f'  {camera.__class__.__name__} não pôde ser inicializada')
        log.error(' TODAS as câmeras falharam!')
        return False

### Método `CameraManager.capture`
O que faz este método? (responda abaixo)


In [ ]:
def capture(self) -> Optional[np.ndarray]:
        if self.active_camera is None:
            raise RuntimeError('Nenhuma câmera inicializada. Chame initialize() primeiro.')
        try:
            image = self.active_camera.capture()
            if image is not None:
                return image
            else:
                log.warning('Câmera ativa retornou None, tentando fallback...')
                raise RuntimeError('Captura falhou')
        except Exception as e:
            log.warning(f' Câmera ativa falhou: {e}')
            return self._try_next_camera()

### Método `CameraManager._try_next_camera`
O que faz este método? (responda abaixo)


In [ ]:
def _try_next_camera(self) -> Optional[np.ndarray]:
        if self._current_index + 1 >= len(self.cameras):
            log.error(' Nenhuma câmera restante na cadeia!')
            return None
        if self.active_camera:
            self.active_camera.release()
        self._current_index += 1
        self.active_camera = self.cameras[self._current_index]
        log.info(f' Alternando para câmera {self._current_index + 1}/{len(self.cameras)}: {self.active_camera.__class__.__name__}')
        try:
            if self.active_camera.initialize():
                return self.active_camera.capture()
            else:
                return self._try_next_camera()
        except Exception as e:
            log.error(f'Falha ao alternar câmera: {e}')
            return self._try_next_camera()

### Método `CameraManager.get_active_camera_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_active_camera_info(self) -> Dict[str, Any]:
        if self.active_camera:
            info = self.active_camera.get_info()
            info['index'] = self._current_index
            info['total_cameras'] = len(self.cameras)
            return info
        return {'status': 'no_active_camera'}

### Método `CameraManager.get_all_cameras_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_all_cameras_info(self) -> List[Dict[str, Any]]:
        infos = []
        for i, camera in enumerate(self.cameras):
            info = camera.get_info()
            info['position'] = i
            info['is_active'] = i == self._current_index
            info['connected'] = camera._initialized if hasattr(camera, '_initialized') else False
            infos.append(info)
        return infos

### Método `CameraManager.detect_all_cameras`
O que faz este método? (responda abaixo)


In [ ]:
def detect_all_cameras(self) -> List[Dict[str, Any]]:
        infos = []
        for i, camera in enumerate(self.cameras):
            camera_info = {'position': i, 'type': camera.__class__.__name__.replace('Camera', '').lower(), 'is_active': i == self._current_index, 'connected': False, 'available': False}
            try:
                if camera.__class__.__name__ == 'MockCamera':
                    camera_info['available'] = True
                    camera_info['connected'] = hasattr(camera, '_initialized') and camera._initialized
                    if camera_info['connected']:
                        camera_info.update(camera.get_info())
                    else:
                        camera_info.update({'model': 'Mock (Simulada)', 'width': 640, 'height': 480})
                elif hasattr(camera, '_initialized') and camera._initialized:
                    camera_info['connected'] = True
                    camera_info['available'] = True
                    camera_info.update(camera.get_info())
                else:
                    if hasattr(camera, 'test_connection'):
                        if camera.test_connection():
                            camera_info['available'] = True
                    elif camera.__class__.__name__ == 'WebcamCamera':
                        camera_info['available'] = True
                    if camera_info['available']:
                        try:
                            camera_info.update(camera.get_info())
                        except:
                            pass
            except Exception as e:
                log.debug(f'Erro ao detectar câmera {i}: {e}')
                camera_info['available'] = False
            infos.append(camera_info)
        return infos

### Método `CameraManager.release`
O que faz este método? (responda abaixo)


In [ ]:
def release(self):
        for camera in self.cameras:
            try:
                camera.release()
            except:
                pass
        self.active_camera = None
        self._current_index = -1
        log.info(' Todas as câmeras liberadas')

## Classe `CameraProfileManager`
O que representa esta classe? (responda abaixo)


### Método `CameraProfileManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, profiles_dir: Optional[Path]=None):
        if profiles_dir is None:
            profiles_dir = Path(__file__).parent.parent.parent / 'data' / 'camera_profiles'
        self.profiles_dir = Path(profiles_dir)
        self.profiles_dir.mkdir(parents=True, exist_ok=True)
        log.info(f'📁 Gerenciador de perfis de câmera: {self.profiles_dir}')

### Método `CameraProfileManager.create_profile`
O que faz este método? (responda abaixo)


In [ ]:
def create_profile(self, profile_name: str, camera_type: str, parameters: Dict[str, Any]) -> bool:
        try:
            if not profile_name or len(profile_name.strip()) == 0:
                log.error('Nome do perfil não pode estar vazio')
                return False
            safe_name = ''.join((c if c.isalnum() or c in '_- ' else '' for c in profile_name))
            if not safe_name:
                log.error('Nome do perfil inválido')
                return False
            profile_path = self.profiles_dir / f'{safe_name}.json'
            if profile_path.exists():
                log.warning(f"Perfil '{safe_name}' já existe")
                return False
            profile_data = {'name': profile_name, 'camera_type': camera_type, 'parameters': parameters, 'created_at': str(Path.cwd() / 'data'), 'timestamp': __import__('datetime').datetime.now().isoformat()}
            with open(profile_path, 'w', encoding='utf-8') as f:
                json.dump(profile_data, f, indent=2, ensure_ascii=False)
            log.info(f'✅ Perfil criado: {profile_name}')
            return True
        except Exception as e:
            log.error(f'❌ Erro ao criar perfil: {e}')
            return False

### Método `CameraProfileManager.load_profile`
O que faz este método? (responda abaixo)


In [ ]:
def load_profile(self, profile_name: str) -> Optional[Dict[str, Any]]:
        try:
            name = profile_name.replace('.json', '').replace('.pfs', '')
            pfs_path = self.profiles_dir / f'{name}.pfs'
            if pfs_path.exists():
                data = {'name': name, 'camera_type': 'basler', 'parameters': {}, '_pfs_file': str(pfs_path), 'format': 'pfs'}
                log.info(f'✅ Perfil .pfs carregado: {name}')
                return data
            json_path = self.profiles_dir / f'{name}.json'
            if json_path.exists():
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                log.info(f'✅ Perfil JSON carregado: {profile_name}')
                return data
            log.warning(f'Perfil não encontrado: {profile_name} (.pfs ou .json)')
            return None
        except Exception as e:
            log.error(f'❌ Erro ao carregar perfil: {e}')
            return None

### Método `CameraProfileManager.delete_profile`
O que faz este método? (responda abaixo)


In [ ]:
def delete_profile(self, profile_name: str) -> bool:
        try:
            name = profile_name.replace('.json', '')
            profile_path = self.profiles_dir / f'{name}.json'
            if not profile_path.exists():
                log.warning(f'Perfil não encontrado: {profile_name}')
                return False
            profile_path.unlink()
            log.info(f'✅ Perfil deletado: {profile_name}')
            return True
        except Exception as e:
            log.error(f'❌ Erro ao deletar perfil: {e}')
            return False

### Método `CameraProfileManager.list_profiles`
O que faz este método? (responda abaixo)


In [ ]:
def list_profiles(self, camera_type: Optional[str]=None) -> List[str]:
        try:
            profiles = []
            seen = set()
            for pfs_file in self.profiles_dir.glob('*.pfs'):
                profile_name = pfs_file.stem
                if profile_name not in seen:
                    if camera_type is None or camera_type == 'basler':
                        profiles.append(profile_name)
                        seen.add(profile_name)
                        log.debug(f'Encontrado perfil .pfs: {profile_name}')
            for json_file in self.profiles_dir.glob('*.json'):
                profile_name = json_file.stem
                if profile_name not in seen:
                    if camera_type:
                        try:
                            with open(json_file, 'r', encoding='utf-8') as f:
                                data = json.load(f)
                            if data.get('camera_type') == camera_type:
                                profiles.append(profile_name)
                                seen.add(profile_name)
                        except:
                            continue
                    else:
                        profiles.append(profile_name)
                        seen.add(profile_name)
            return sorted(profiles)
        except Exception as e:
            log.error(f'❌ Erro ao listar perfis: {e}')
            return []

### Método `CameraProfileManager.profile_exists`
O que faz este método? (responda abaixo)


In [ ]:
def profile_exists(self, profile_name: str) -> bool:
        name = profile_name.replace('.json', '')
        return (self.profiles_dir / f'{name}.json').exists()

### Método `CameraProfileManager.rename_profile`
O que faz este método? (responda abaixo)


In [ ]:
def rename_profile(self, old_name: str, new_name: str) -> bool:
        try:
            old_path = self.profiles_dir / f'{old_name}.json'
            new_path = self.profiles_dir / f'{new_name}.json'
            if not old_path.exists():
                log.warning(f'Perfil não encontrado: {old_name}')
                return False
            if new_path.exists():
                log.warning(f'Perfil já existe: {new_name}')
                return False
            with open(old_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            data['name'] = new_name
            with open(new_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            old_path.unlink()
            log.info(f'✅ Perfil renomeado: {old_name} → {new_name}')
            return True
        except Exception as e:
            log.error(f'❌ Erro ao renomear perfil: {e}')
            return False

## Classe `ICamera`
O que representa esta classe? (responda abaixo)


### Método `ICamera.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        pass

### Método `ICamera.capture`
O que faz este método? (responda abaixo)


In [ ]:
def capture(self) -> Optional[np.ndarray]:
        pass

### Método `ICamera.release`
O que faz este método? (responda abaixo)


In [ ]:
def release(self):
        pass

### Método `ICamera.get_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_info(self) -> Dict[str, Any]:
        pass

### Método `ICamera.is_available`
O que faz este método? (responda abaixo)


In [ ]:
def is_available(self) -> bool:
        pass

### Método `ICamera.get_parameters`
O que faz este método? (responda abaixo)


In [ ]:
def get_parameters(self) -> Dict[str, Any]:
        return {}

### Método `ICamera.set_parameter`
O que faz este método? (responda abaixo)


In [ ]:
def set_parameter(self, param_name: str, value: Any) -> bool:
        return False

## Classe `MockCamera`
O que representa esta classe? (responda abaixo)


### Método `MockCamera.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, image_path: Optional[str]=None):
        self.image_path = image_path or str(ASSETS_DIR / 'test_image.jpg')
        self._image = None
        self._initialized = False
        log.info(f' MockCamera criada (imagem: {self.image_path})')

### Método `MockCamera.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        try:
            path = Path(self.image_path)
            if not path.exists():
                log.warning(f'Imagem não encontrada: {self.image_path}')
                log.info('Criando imagem fake para teste...')
                self._image = np.zeros((480, 640, 3), dtype=np.uint8)
                cv2.putText(self._image, 'MOCK CAMERA', (50, 240), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
                cv2.putText(self._image, 'Sistema em Desenvolvimento', (30, 300), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                log.info(' Imagem fake criada')
            else:
                self._image = cv2.imread(str(path))
                if self._image is None:
                    raise ValueError(f'Não foi possível ler imagem: {path}')
                log.info(f' Imagem carregada: {self._image.shape}')
            self._initialized = True
            return True
        except Exception as e:
            log.error(f' Erro ao inicializar MockCamera: {e}')
            return False

### Método `MockCamera.capture`
O que faz este método? (responda abaixo)


In [ ]:
def capture(self) -> Optional[np.ndarray]:
        if not self._initialized:
            raise RuntimeError('MockCamera não inicializada')
        log.debug(' MockCamera: retornando imagem simulada')
        return self._image.copy()

### Método `MockCamera.release`
O que faz este método? (responda abaixo)


In [ ]:
def release(self):
        self._image = None
        self._initialized = False
        log.debug(' MockCamera liberada')

### Método `MockCamera.get_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_info(self) -> Dict[str, Any]:
        info = {'type': 'mock', 'image_path': self.image_path, 'initialized': self._initialized, 'note': 'Câmera simulada para desenvolvimento'}
        if self._image is not None:
            info['image_shape'] = self._image.shape
            info['image_dtype'] = str(self._image.dtype)
        return info

### Método `MockCamera.is_available`
O que faz este método? (responda abaixo)


In [ ]:
def is_available(self) -> bool:
        return True

## Classe `WebcamCamera`
O que representa esta classe? (responda abaixo)


### Método `WebcamCamera.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, camera_index: int=0):
        self.camera_index = camera_index
        self._cap = None
        self._initialized = False
        log.info(f' Webcam criada (índice: {camera_index})')

### Método `WebcamCamera.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        try:
            log.info(f'Abrindo webcam {self.camera_index}...')
            self._cap = cv2.VideoCapture(self.camera_index)
            if not self._cap.isOpened():
                log.error(f'Webcam {self.camera_index} não pôde ser aberta')
                return False
            self._cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
            self._cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
            ret, test_frame = self._cap.read()
            if not ret:
                log.error('Webcam aberta mas não consegue capturar')
                self._cap.release()
                return False
            self._initialized = True
            log.info(f' Webcam {self.camera_index} inicializada')
            width = int(self._cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(self._cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            log.info(f'  Resolução: {width}x{height}')
            return True
        except Exception as e:
            log.error(f' Erro ao inicializar webcam: {e}')
            return False

### Método `WebcamCamera.capture`
O que faz este método? (responda abaixo)


In [ ]:
def capture(self) -> Optional[np.ndarray]:
        if not self._initialized or self._cap is None:
            raise RuntimeError('Webcam não inicializada')
        try:
            ret, frame = self._cap.read()
            if ret and frame is not None:
                log.debug(f' Webcam capturou: {frame.shape}')
                return frame
            else:
                log.warning('Webcam não retornou frame')
                return None
        except Exception as e:
            log.error(f'Erro na captura da webcam: {e}')
            return None

### Método `WebcamCamera.release`
O que faz este método? (responda abaixo)


In [ ]:
def release(self):
        if self._cap is not None:
            self._cap.release()
            log.info(' Webcam liberada')
        self._initialized = False

### Método `WebcamCamera.get_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_info(self) -> Dict[str, Any]:
        info = {'type': 'webcam', 'index': self.camera_index, 'initialized': self._initialized}
        if self._cap is not None:
            info['width'] = int(self._cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            info['height'] = int(self._cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            info['fps'] = self._cap.get(cv2.CAP_PROP_FPS)
        return info

### Método `WebcamCamera.is_available`
O que faz este método? (responda abaixo)


In [ ]:
def is_available(self) -> bool:
        cap = cv2.VideoCapture(self.camera_index)
        if cap.isOpened():
            cap.release()
            return True
        return False

### Método `WebcamCamera.get_parameters`
O que faz este método? (responda abaixo)


In [ ]:
def get_parameters(self) -> Dict[str, Any]:
        if not self._initialized or self._cap is None:
            return {}
        return {'brightness': {'value': self._cap.get(cv2.CAP_PROP_BRIGHTNESS), 'type': 'float', 'label': 'Brilho', 'min': -100, 'max': 100, 'step': 5, 'description': 'Ajusta brilho da imagem (-100 a 100)'}, 'contrast': {'value': self._cap.get(cv2.CAP_PROP_CONTRAST), 'type': 'float', 'label': 'Contraste', 'min': 0, 'max': 100, 'step': 5, 'description': 'Ajusta contraste da imagem (0 a 100)'}, 'saturation': {'value': self._cap.get(cv2.CAP_PROP_SATURATION), 'type': 'float', 'label': 'Saturação', 'min': 0, 'max': 100, 'step': 5, 'description': 'Ajusta saturação de cores (0 a 100)'}, 'hue': {'value': self._cap.get(cv2.CAP_PROP_HUE), 'type': 'float', 'label': 'Matiz', 'min': -180, 'max': 180, 'step': 10, 'description': 'Rotação de cor em graus'}, 'exposure': {'value': self._cap.get(cv2.CAP_PROP_EXPOSURE), 'type': 'float', 'label': 'Exposição', 'min': -13, 'max': 0, 'step': 1, 'description': 'Ajusta tempo de exposição (log scale, -13 a 0)'}, 'gain': {'value': self._cap.get(cv2.CAP_PROP_GAIN), 'type': 'float', 'label': 'Ganho', 'min': 0, 'max': 100, 'step': 5, 'description': 'Ganho do sensor (0 a 100)'}, 'width': {'value': int(self._cap.get(cv2.CAP_PROP_FRAME_WIDTH)), 'type': 'int', 'label': 'Largura', 'min': 320, 'max': 3840, 'step': 160, 'description': 'Largura da imagem em pixels'}, 'height': {'value': int(self._cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), 'type': 'int', 'label': 'Altura', 'min': 240, 'max': 2160, 'step': 120, 'description': 'Altura da imagem em pixels'}}

### Método `WebcamCamera.set_parameter`
O que faz este método? (responda abaixo)


In [ ]:
def set_parameter(self, param_name: str, value: Any) -> bool:
        if not self._initialized or self._cap is None:
            return False
        try:
            param_map = {'brightness': cv2.CAP_PROP_BRIGHTNESS, 'contrast': cv2.CAP_PROP_CONTRAST, 'saturation': cv2.CAP_PROP_SATURATION, 'hue': cv2.CAP_PROP_HUE, 'exposure': cv2.CAP_PROP_EXPOSURE, 'gain': cv2.CAP_PROP_GAIN, 'width': cv2.CAP_PROP_FRAME_WIDTH, 'height': cv2.CAP_PROP_FRAME_HEIGHT}
            if param_name in param_map:
                self._cap.set(param_map[param_name], float(value))
                log.info(f' Webcam: {param_name} = {value}')
                return True
            return False
        except Exception as e:
            log.error(f'Erro ao ajustar webcam: {e}')
            return False

## 🧹 LIMPEZA DE MEMÓRIA

In [ ]:
import gc

# Limpar cache e liberar memória
try:
    del model
    print("✓ Modelo deletado")
except NameError:
    print("ℹ Nenhum modelo em memória")

gc.collect()
print("✓ Garbage collection executado")